# Zero-shot detectors on a GPU: Binoculars, Fast-DetectGPT, log-likelihood, log-rank

Before running: **Runtime > Change runtime type > GPU**.

| Pair | Models | GPU needed |
|---|---|---|
| `qwen2.5-1.5b` | Qwen2.5-1.5B + Qwen2.5-1.5B-Instruct | T4 (free tier) |
| `falcon-7b` | Falcon-7B + Falcon-7B-Instruct (official Binoculars pair) | A100 40 GB |

The notebook rebuilds the same frozen dataset as on your machine (same `content_sha256`),
scores the calibration and test splits, evaluates them and gives you a zip to download.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO = "https://github.com/Azekf11/IA_Writing.git"
PAIR = "qwen2.5-1.5b"  # or "falcon-7b" on an A100
![ -d /content/IA_Writing ] || git clone -q {REPO} /content/IA_Writing
%cd /content/IA_Writing/backend
%pip install -q -e ".[ml]"

Download RAID (`train_none.csv`, about 802 MB) and rebuild the dataset.
Check that the printed `content_sha256` is the one in your local
`backend/data/datasets/raid-v1.manifest.json`: if it differs, stop and compare versions.

In [ ]:
!python -c "from aiwd.data.raid import download_raid; print(download_raid())"
!python scripts/build_dataset.py | tail -2

Smoke test on 40 texts first: it downloads the two models and checks that the scores are finite.
If it fails with "non-finite logits", add `--dtype bfloat16` to the commands below
(same memory as float16, slower on a T4).

In [ ]:
!python scripts/score_zeroshot.py --pair {PAIR} --limit 40 --batch-size 8

Full run: every human text of the calibration and test splits, plus a deterministic 25%
sample of the test AI texts (proportions of domains and generators kept). The longest texts
go first, so an out-of-memory error shows up within a minute: lower `--batch-size` to 4 or 2.
Expect very roughly 15 to 40 minutes on a T4 with `qwen2.5-1.5b`.

In [ ]:
!python scripts/score_zeroshot.py --pair {PAIR} --ai-fraction 0.25 --batch-size 8

In [ ]:
for criterion in ["binoculars", "fast-detectgpt", "loglik", "logrank"]:
    name = f"{criterion}-{PAIR}"
    cmd = f"python scripts/run_eval.py --scores data/scores/{name}.parquet --name {name}"
    !{cmd} --ai-fraction 0.25 | sed -n 1,14p

Download the results, then on your Mac:
`unzip -o ~/Downloads/zeroshot-<pair>.zip -d ~/IA_Writing/backend` and commit `backend/reports`.

In [ ]:
!zip -qr /content/zeroshot-{PAIR}.zip data/features data/scores reports
from google.colab import files

files.download(f"/content/zeroshot-{PAIR}.zip")